<a href="https://colab.research.google.com/github/Ash100/Python_for_Lifescience/blob/main/Chapter_6%3APandas_for_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab에서 열기"/></a>


# 6장: 데이터 분석을 위한 Pandas


### 소개
구조화된 데이터 작업을 위한 강력한 파이썬 라이브러리인 **Pandas**에 대한 이 장에 오신 것을 환영합니다. 생물학에서는 유전자 발현 행렬, 단백질 상호 작용 표, 임상 시험 결과 등과 같은 표 형식 데이터를 자주 다룹니다. Pandas는 이러한 데이터를 효율적으로 정리, 필터링 및 분석하는 데 도움이 됩니다.

이 장에서는 다음을 배우게 됩니다:
- 데이터프레임 생성 및 조작
- 생물학적 데이터셋 정리
- 실험 결과 필터링 및 그룹화
- 결측 데이터 처리

시작해 봅시다!


### 🔧 1단계: 설정

시작하기 전에 사용할 라이브러리를 설치하고 가져와야 합니다.  
- `pandas`: 데이터 조작용  
- `seaborn`: 시각화용  
- `numpy`: 수치 연산용


In [ ]:
# Install necessary libraries
# !pip install pandas seaborn

# Import libraries
import pandas as pd

## 구조화된 생물학적 데이터를 위한 데이터프레임(df)

### 🧬 데이터프레임이란 무엇입니까?

**데이터프레임**은 파이썬의 엑셀 스프레드시트와 같습니다. 각 행은 관찰(예: 유전자)이고 각 열은 변수(예: 발현 수준, 조직 유형)입니다.

여기서는 유전자 발현 데이터셋을 시뮬레이션합니다:
- `Gene`: 유전자 이름
- `Expression_Level`: 측정된 발현
- `Tissue`: 측정된 조직
- `Condition`: 샘플이 건강한지 또는 암인지 여부

이 구조는 생물정보학 및 실험 생물학에서 일반적입니다.


In [3]:
# Sample gene expression dataset
data = {
    "Gene": ["TP53", "BRCA1", "EGFR", "MYC", "CDK2"],
    "Expression_Level": [7.2, 5.5, 8.1, 6.3, 4.9],
    "Tissue": ["Liver", "Breast", "Lung", "Brain", "Liver"],
    "Condition": ["Healthy", "Cancer", "Cancer", "Healthy", "Cancer"],
}

df = pd.DataFrame(data)
df

,Gene,Expression_Level,Tissue,Condition
0,TP53,7.2,Liver,Healthy
1,BRCA1,5.5,Breast,Cancer
2,EGFR,8.1,Lung,Cancer
3,MYC,6.3,Brain,Healthy
4,CDK2,4.9,Liver,Cancer


## 🧹 생물학적 데이터 정리/처리

실제 생물학적 데이터는 지저분합니다. 종종 다음을 수행해야 합니다:
- 명확성을 위해 열 이름 바꾸기
- 값 정규화(예: 최대 발현으로 나누기)
- 레이블 표준화(예: "Liver" → "Hepatic")

이러한 단계는 다운스트림 분석을 더 쉽고 재현 가능하게 만듭니다.


In [4]:
# Rename columns for clarity
df.rename(columns={"Expression_Level": "Expr"}, inplace=True)

# Add a normalized expression column
df["Expr_Norm"] = df["Expr"] / df["Expr"].max()

# Replace tissue names for consistency
df["Tissue"] = df["Tissue"].replace({"Liver": "Hepatic"})

df

,Gene,Expr,Tissue,Condition,Expr_Norm
0,TP53,7.2,Hepatic,Healthy,0.888889
1,BRCA1,5.5,Breast,Cancer,0.679012
2,EGFR,8.1,Lung,Cancer,1.000000
3,MYC,6.3,Brain,Healthy,0.777778
4,CDK2,4.9,Hepatic,Cancer,0.604938


## 🔬 필터링 및 그룹화

우리는 종종 다음을 원합니다:
- 특정 조건(예: 암 샘플)에 집중
- 그룹 비교(예: 조직)

여기서는 다음을 수행합니다:
- `Condition == 'Cancer'`인 행 필터링
- `Tissue`별로 그룹화하고 평균 및 최대 발현 계산

이는 실험 반복 또는 조건을 비교하는 것을 모방합니다.


In [5]:
# Filter for cancer samples
cancer_df = df.query("Condition == 'Cancer'")
cancer_df

,Gene,Expr,Tissue,Condition,Expr_Norm
1,BRCA1,5.5,Breast,Cancer,0.679012
2,EGFR,8.1,Lung,Cancer,1.000000
4,CDK2,4.9,Hepatic,Cancer,0.604938


In [6]:
# Group by tissue and compute mean/max expression
grouped = df.groupby("Tissue").agg({"Expr": ["mean", "max"]})
grouped

Expr     
         mean  max
Tissue            
Brain    6.30  6.3
Breast   5.50  5.5
Hepatic  6.05  7.2
Lung     8.10  8.1

## ❗ 생물학의 결측 데이터

결측 데이터는 실패한 실험, 읽을 수 없는 파일 또는 낮은 신호로 인해 흔히 발생합니다.

다음을 수행할 수 있습니다:
- `.isna()`를 사용하여 결측값 감지
- 통계적 추정치(예: 평균)로 채우기
- 필요한 경우 삭제

결측 데이터를 처리하는 방법을 항상 문서화하십시오. 이는 생물학적 해석에 영향을 미칩니다. <br>
**먼저 결측 데이터가 있는 df를 만들어 보겠습니다.**


In [7]:
# Create a DataFrame with missing data
missing_data = {
    "SampleID": ["P1", "P2", "P3", "P4"],
    "Age": [35, 42, None, 58],
    "BiomarkerA": [1.2, 1.5, 1.1, None],
    "BiomarkerB": [15.2, 16.5, 14.8, 17.1],
}
missing_df = pd.DataFrame(missing_data)
print("Original DataFrame with Missing Data:")
print(missing_df)

# Check for missing values
print("\nMissing values per column:")
print(missing_df.isnull().sum())

Original DataFrame with Missing Data:
  SampleID   Age  BiomarkerA  BiomarkerB
0       P1  35.0         1.2        15.2
1       P2  42.0         1.5        16.5
2       P3   NaN         1.1        14.8
3       P4  58.0         NaN        17.1

Missing values per column:
SampleID      0
Age           1
BiomarkerA    1
BiomarkerB    0
dtype: int64


**결측 데이터 삭제**<br>
`.dropna()` 메서드는 결측값이 있는 행 또는 열을 제거합니다.


In [8]:
# Create a new DataFrame by dropping all rows with any missing values
df_dropped = missing_df.dropna()
print("DataFrame after dropping rows with missing values:")
print(df_dropped)

DataFrame after dropping rows with missing values:
  SampleID   Age  BiomarkerA  BiomarkerB
0       P1  35.0         1.2        15.2
1       P2  42.0         1.5        16.5


**결측 데이터 채우기**<br>
`.fillna()` 메서드는 결측값을 지정된 값으로 바꿉니다. 일반적인 접근 방식은 결측값을 열의 평균 또는 중앙값으로 채우는 것입니다.


In [9]:
# Calculate the mean age and median of BiomarkerA
mean_age = missing_df["Age"].mean()
median_biomarker_a = missing_df["BiomarkerA"].median()

# Fill missing 'Age' with the mean and missing 'BiomarkerA' with the median
df_filled = missing_df.fillna(value={"Age": mean_age, "BiomarkerA": median_biomarker_a})
print("DataFrame after filling missing values:")
print(df_filled)

DataFrame after filling missing values:
  SampleID   Age  BiomarkerA  BiomarkerB
0       P1  35.0         1.2        15.2
1       P2  42.0         1.5        16.5
2       P3  45.0         1.1        14.8
3       P4  58.0         1.2        17.1


여러분이 새로운 것을 배우고 있기를 바랍니다. 포기하지 말고 계속 연습하십시오.
